# 🚀 Run RAG (Naive vs Advanced) + RAGAS Eval on Google Colab

A self-contained launcher for the standalone scripts (`common.py`,
`naive_rag.py`, `advanced_rag.py`, `evaluate.py`).

**What this notebook does**
1. Writes the 4 script files into Colab's `/content/` filesystem
2. Installs the pinned dependency set
3. Loads your `GROQ_API_KEY` from Colab **Secrets** (🔑 icon in the left sidebar)
4. Runs the Naive RAG demo, the Advanced RAG demo, and finally the full RAGAS evaluation
5. Shows the resulting bar chart inline

---

## Prerequisites (1 min)

| Step | How |
|---|---|
| 1. Free Groq API key | https://console.groq.com/keys |
| 2. Add it to Colab secrets | Left sidebar 🔑 → **Add new secret** → Name = `GROQ_API_KEY` → paste your key → toggle **Notebook access** ON |
| 3. (Optional) T4 GPU | **Runtime → Change runtime type → T4 GPU** — makes the cross-encoder ~10× faster |

> If you enable the GPU, change `reranker_device="cpu"` → `"cuda"` in Step 3 below.


## Step 1 — Materialise the scripts into `/content/`

In [ ]:
%%writefile common.py
"""
common.py — shared utilities for the RAG scripts.

Provides:
    - load_groq_key()          portable GROQ_API_KEY loader (env / .env / Colab / prompt)
    - make_llm_client()        thin wrapper returning a Groq client + call_llm(prompt)
    - download_pdf(url, path)  idempotent PDF download
    - load_and_chunk_pdf(...)  PyMuPDF loader + RecursiveCharacterTextSplitter
    - build_faiss(docs)        FAISS vector store from HuggingFace MiniLM embeddings
    - preprocess(text)         lowercase + strip punctuation → tokens (for BM25)
    - RAG_PROMPT               canonical grounded-answer prompt template

Runnable smoke test:
    python common.py
"""

from __future__ import annotations

import os
import re
import tempfile
import urllib.request
from getpass import getpass
from typing import Callable, List, Tuple

from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter


# ── Constants ────────────────────────────────────────────────────────
DEFAULT_PDF_URL = "https://arxiv.org/pdf/1706.03762.pdf"
DEFAULT_PDF_NAME = "Attention is all you need.pdf"
DEFAULT_EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
DEFAULT_LLM_MODEL = "llama-3.1-8b-instant"

RAG_PROMPT = """You are a precise assistant. Answer the question using ONLY the context below.
If the answer is not in the context, say: "Not found in document".
Cite pages like [Page X].

Context:
{context}

Question: {question}

Answer:"""


# ── API-key loader ───────────────────────────────────────────────────
def load_groq_key() -> str:
    """Load GROQ_API_KEY from env / Colab secrets / .env / interactive prompt.

    Sets os.environ["GROQ_API_KEY"] as a side effect.
    Returns the source it was loaded from.
    """
    if os.environ.get("GROQ_API_KEY"):
        return "environment"
    try:
        from google.colab import userdata  # type: ignore
        os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
        return "colab-secrets"
    except Exception:
        pass
    try:
        from dotenv import load_dotenv
        if load_dotenv() and os.environ.get("GROQ_API_KEY"):
            return "dotenv"
    except ImportError:
        pass
    os.environ["GROQ_API_KEY"] = getpass("Enter your GROQ_API_KEY: ").strip()
    return "prompt"


# ── LLM client factory ───────────────────────────────────────────────
def make_llm_client(model: str = DEFAULT_LLM_MODEL, temperature: float = 0.2):
    """Return (client, call_llm) — call_llm(prompt) → str."""
    from groq import Groq

    if not os.environ.get("GROQ_API_KEY"):
        load_groq_key()
    client = Groq(api_key=os.environ["GROQ_API_KEY"])

    def call_llm(prompt: str) -> str:
        res = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            temperature=temperature,
        )
        return res.choices[0].message.content

    return client, call_llm


# ── PDF download ─────────────────────────────────────────────────────
def download_pdf(url: str = DEFAULT_PDF_URL, name: str = DEFAULT_PDF_NAME) -> str:
    """Download `url` to a stable OS temp location if missing. Return the file path."""
    if os.path.isdir("/content"):  # Colab
        pdf_dir = "/content"
    else:
        pdf_dir = os.path.join(tempfile.gettempdir(), "rag_demo")
        os.makedirs(pdf_dir, exist_ok=True)

    pdf_path = os.path.join(pdf_dir, name)
    if not os.path.exists(pdf_path):
        print(f"Downloading PDF → {pdf_path} …")
        urllib.request.urlretrieve(url, pdf_path)
    return pdf_path


# ── PDF loading + chunking ───────────────────────────────────────────
def load_and_chunk_pdf(
    pdf_path: str,
    chunk_size: int = 1000,
    chunk_overlap: int = 200,
) -> Tuple[List[Document], List[Document]]:
    """Return (pages, chunks). Chunks get chunk_id + 1-indexed page metadata."""
    pages = PyMuPDFLoader(pdf_path).load()
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunk_overlap
    )
    docs = splitter.split_documents(pages)
    for i, d in enumerate(docs):
        d.metadata["chunk_id"] = i
        d.metadata["page"] = d.metadata.get("page", 0) + 1
    return pages, docs


# ── Dense embeddings + FAISS ─────────────────────────────────────────
def build_faiss(
    docs: List[Document],
    embed_model: str = DEFAULT_EMBED_MODEL,
) -> Tuple[HuggingFaceEmbeddings, FAISS]:
    """Return (embeddings, vectorstore)."""
    embeddings = HuggingFaceEmbeddings(model_name=embed_model)
    vectorstore = FAISS.from_documents(docs, embeddings)
    return embeddings, vectorstore


# ── BM25 preprocessing ───────────────────────────────────────────────
def preprocess(text: str) -> List[str]:
    """Lowercase, drop punctuation, split on whitespace. Used for BM25 tokens."""
    text = text.lower()
    text = re.sub(r"[^\w\s]", " ", text)
    return text.split()


# ── Context builder for prompts ──────────────────────────────────────
def build_context(chunks: List[Document]) -> str:
    """Concatenate chunk texts with page-tagged headers."""
    return "\n\n---\n\n".join(
        f"[Page {d.metadata.get('page', '?')}]\n{d.page_content}" for d in chunks
    )


# ── CLI smoke test ───────────────────────────────────────────────────
def _smoke_test() -> None:
    print(f"✅ GROQ_API_KEY loaded from: {load_groq_key()}")
    pdf_path = download_pdf()
    pages, chunks = load_and_chunk_pdf(pdf_path)
    print(f"✅ Loaded {len(pages)} pages · {len(chunks)} chunks")
    _, vs = build_faiss(chunks)
    hits = vs.similarity_search("What is attention?", k=3)
    print(f"✅ FAISS returned {len(hits)} chunks for sample query")
    _, call_llm = make_llm_client()
    reply = call_llm("Say 'ready' and nothing else.")
    print(f"✅ Groq LLM reply: {reply!r}")


if __name__ == "__main__":
    _smoke_test()


In [ ]:
%%writefile naive_rag.py
"""
naive_rag.py — Naive RAG pipeline (PDF → chunk → embed → FAISS → LLM).

Exposes:
    NaiveRAG.build()      one-shot factory: downloads PDF, chunks, indexes
    NaiveRAG.query(q, k)  returns {"answer", "retrieved_chunks", "pages_used"}

Run as a script:
    python naive_rag.py
"""

from __future__ import annotations

from dataclasses import dataclass
from typing import Callable, Dict, List

from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

from common import (
    RAG_PROMPT,
    build_context,
    build_faiss,
    download_pdf,
    load_and_chunk_pdf,
    make_llm_client,
)


@dataclass
class NaiveRAG:
    """PDF → FAISS (dense) → LLM. The baseline pipeline."""

    docs: List[Document]
    vectorstore: FAISS
    call_llm: Callable[[str], str]

    @classmethod
    def build(cls, pdf_path: str | None = None) -> "NaiveRAG":
        """One-shot factory: downloads the PDF if needed, chunks, and indexes."""
        pdf_path = pdf_path or download_pdf()
        _, docs = load_and_chunk_pdf(pdf_path)
        _, vectorstore = build_faiss(docs)
        _, call_llm = make_llm_client()
        return cls(docs=docs, vectorstore=vectorstore, call_llm=call_llm)

    def query(self, question: str, k: int = 5) -> Dict:
        retrieved = self.vectorstore.similarity_search(question, k=k)
        context = build_context(retrieved)
        answer = self.call_llm(RAG_PROMPT.format(context=context, question=question))
        return {
            "answer": answer,
            "retrieved_chunks": retrieved,
            "pages_used": sorted({d.metadata.get("page", "?") for d in retrieved}),
        }


# ── CLI demo ─────────────────────────────────────────────────────────
DEMO_QUERIES = [
    "What is the main idea of this paper?",
    "What is the Transformer architecture?",
    "What datasets were used in the experiments?",
    "Who are the authors of this paper?",  # expected to fail — motivates BM25
]


def main() -> None:
    print("🔧 Building Naive RAG pipeline …")
    rag = NaiveRAG.build()
    print(f"✅ Ready — {len(rag.docs)} chunks indexed\n")

    for q in DEMO_QUERIES:
        print("═" * 70)
        print(f" Question: {q}")
        print("═" * 70)
        out = rag.query(q)
        print(f" Pages used: {out['pages_used']}")
        print(f"\n Answer:\n{out['answer']}\n")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile advanced_rag.py
"""
advanced_rag.py — Hybrid retrieval (FAISS + BM25) → RRF fusion → Cross-Encoder rerank → LLM.

Exposes:
    AdvancedRAG.build()      one-shot factory
    AdvancedRAG.query(q, top_k, fetch_k)
    rrf(result_lists, k)     reusable Reciprocal Rank Fusion

Run as a script:
    python advanced_rag.py
"""

from __future__ import annotations

from dataclasses import dataclass
from typing import Callable, Dict, List

import numpy as np
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder

from common import (
    RAG_PROMPT,
    build_context,
    build_faiss,
    download_pdf,
    load_and_chunk_pdf,
    make_llm_client,
    preprocess,
)


# ── Reciprocal Rank Fusion ───────────────────────────────────────────
def rrf(result_lists: List[List[Document]], k: int = 60) -> List[Dict]:
    """Reciprocal Rank Fusion: score = Σ 1 / (k + rank + 1). Returns descending list."""
    scores: Dict[int, Dict] = {}
    for results in result_lists:
        for rank, doc in enumerate(results):
            key = hash(doc.page_content)
            scores.setdefault(key, {"doc": doc, "score": 0.0})
            scores[key]["score"] += 1.0 / (rank + k + 1)
    return sorted(scores.values(), key=lambda x: x["score"], reverse=True)


@dataclass
class AdvancedRAG:
    """Hybrid retrieval + RRF + Cross-Encoder rerank → LLM."""

    docs: List[Document]
    vectorstore: FAISS
    bm25: BM25Okapi
    reranker: CrossEncoder
    call_llm: Callable[[str], str]

    @classmethod
    def build(
        cls,
        pdf_path: str | None = None,
        reranker_model: str = "cross-encoder/ms-marco-MiniLM-L-6-v2",
        reranker_device: str = "cpu",
    ) -> "AdvancedRAG":
        pdf_path = pdf_path or download_pdf()
        _, docs = load_and_chunk_pdf(pdf_path)
        _, vectorstore = build_faiss(docs)
        bm25 = BM25Okapi([preprocess(d.page_content) for d in docs])
        reranker = CrossEncoder(reranker_model, device=reranker_device)
        _, call_llm = make_llm_client()
        return cls(
            docs=docs,
            vectorstore=vectorstore,
            bm25=bm25,
            reranker=reranker,
            call_llm=call_llm,
        )

    def query(self, question: str, top_k: int = 5, fetch_k: int = 20) -> Dict:
        # 1. DENSE retrieval
        sem_docs = self.vectorstore.similarity_search(question, k=fetch_k)

        # 2. SPARSE retrieval (BM25)
        bm_scores = self.bm25.get_scores(preprocess(question))
        bm_idx = np.argsort(bm_scores)[::-1][:fetch_k]
        kw_docs = [self.docs[i] for i in bm_idx]

        # 3. RRF fusion
        fused = [x["doc"] for x in rrf([sem_docs, kw_docs])]

        # 4. Cross-encoder rerank
        pairs = [(question, d.page_content) for d in fused]
        ce_scores = self.reranker.predict(pairs)
        ranked = sorted(zip(fused, ce_scores), key=lambda x: x[1], reverse=True)
        top_docs = [d for d, _ in ranked[:top_k]]

        # 5. Build prompt & call LLM
        context = build_context(top_docs)
        answer = self.call_llm(RAG_PROMPT.format(context=context, question=question))
        return {
            "answer": answer,
            "retrieved_chunks": top_docs,
            "pages_used": sorted({d.metadata.get("page", "?") for d in top_docs}),
        }


# ── CLI demo ─────────────────────────────────────────────────────────
DEMO_QUERIES = [
    "What is the main idea of this paper?",
    "What is the Transformer architecture?",
    "What datasets were used in the experiments?",
    "Who are the authors of this paper?",  # now succeeds thanks to BM25
]


def main() -> None:
    print("🔧 Building Advanced RAG pipeline …")
    rag = AdvancedRAG.build()
    print(f"✅ Ready — {len(rag.docs)} chunks · FAISS + BM25 + Cross-Encoder\n")

    for q in DEMO_QUERIES:
        print("═" * 70)
        print(f" Question: {q}")
        print("═" * 70)
        out = rag.query(q)
        print(f" Pages used: {out['pages_used']}")
        print(f"\n Answer:\n{out['answer']}\n")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile evaluate.py
"""
evaluate.py — RAGAS evaluation of Naive vs Advanced RAG.

Runs both pipelines on a small golden test set, grades them with 4 RAGAS
metrics (Faithfulness · Answer Relevancy · Context Precision · Context Recall)
using a Groq Llama judge, prints per-pipeline averages + % lift, and saves
optional bar chart + CSV outputs.

Run:
    python evaluate.py
    python evaluate.py --judge llama-3.3-70b-versatile --outdir results/
"""

from __future__ import annotations

import argparse
import os
from pathlib import Path
from typing import Dict, List

import pandas as pd

from advanced_rag import AdvancedRAG
from common import DEFAULT_EMBED_MODEL, load_groq_key
from naive_rag import NaiveRAG


# ── Golden test set (question + ground-truth answer) ─────────────────
TEST_SET: List[Dict[str, str]] = [
    {
        "question": "What is the main idea of this paper?",
        "ground_truth": (
            "The paper proposes the Transformer, a new network architecture based "
            "solely on attention mechanisms, dispensing entirely with recurrence "
            "and convolutions. It achieves superior translation quality while "
            "being more parallelizable and faster to train."
        ),
    },
    {
        "question": "What is the Transformer architecture?",
        "ground_truth": (
            "The Transformer follows an encoder-decoder structure using stacked "
            "self-attention and point-wise fully connected layers for both the "
            "encoder and decoder, as shown in Figure 1."
        ),
    },
    {
        "question": "What datasets were used in the experiments?",
        "ground_truth": (
            "WMT 2014 English-German (about 4.5M sentence pairs), "
            "WMT 2014 English-French (36M sentences), "
            "Wall Street Journal portion of the Penn Treebank (~40K sentences), "
            "and high-confidence + BerkeleyParser corpora (~17M sentences)."
        ),
    },
    {
        "question": "Who are the authors of this paper?",
        "ground_truth": (
            "Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, "
            "Llion Jones, Aidan N. Gomez, Łukasz Kaiser, and Illia Polosukhin."
        ),
    },
]


def run_pipeline(rag, testset: List[Dict[str, str]], label: str) -> Dict[str, list]:
    """Run one RAG pipeline over the test set. Returns RAGAS-shaped dict."""
    print(f"\n▶ Running {label} …")
    rows: Dict[str, list] = {
        "question": [],
        "answer": [],
        "contexts": [],
        "ground_truth": [],
    }
    for i, item in enumerate(testset, 1):
        print(f"   [{i}/{len(testset)}] {item['question'][:60]}…")
        out = rag.query(item["question"])
        rows["question"].append(item["question"])
        rows["answer"].append(out["answer"])
        rows["contexts"].append([d.page_content for d in out["retrieved_chunks"]])
        rows["ground_truth"].append(item["ground_truth"])
    return rows


def build_ragas_judge(judge_model: str, embeddings):
    """Return (judge_llm, judge_embeddings, metrics) tuple for ragas.evaluate."""
    from langchain_groq import ChatGroq
    from ragas.metrics import (
        answer_relevancy,
        context_precision,
        context_recall,
        faithfulness,
    )

    try:
        from ragas.llms import LangchainLLMWrapper
        from ragas.embeddings import LangchainEmbeddingsWrapper
    except ImportError:  # ragas >= 0.2 alt layout
        from ragas.llms.base import LangchainLLMWrapper
        from ragas.embeddings.base import LangchainEmbeddingsWrapper

    try:
        chat = ChatGroq(
            model=judge_model, temperature=0, api_key=os.environ["GROQ_API_KEY"]
        )
    except TypeError:
        chat = ChatGroq(
            model=judge_model, temperature=0, groq_api_key=os.environ["GROQ_API_KEY"]
        )

    judge_llm = LangchainLLMWrapper(chat)
    judge_embeddings = LangchainEmbeddingsWrapper(embeddings)
    metrics = [faithfulness, answer_relevancy, context_precision, context_recall]
    return judge_llm, judge_embeddings, metrics


def evaluate_pipeline(rows, label, judge_llm, judge_embeddings, metrics) -> pd.DataFrame:
    from datasets import Dataset
    from ragas import evaluate

    print(f"\n  Evaluating {label} (4 metrics × {len(rows['question'])} questions)…")
    ds = Dataset.from_dict(rows)
    result = evaluate(ds, metrics=metrics, llm=judge_llm, embeddings=judge_embeddings)
    df = result.to_pandas()
    df.insert(0, "pipeline", label)
    return df


def print_summary(df_all: pd.DataFrame) -> pd.DataFrame:
    metric_cols = [
        "faithfulness",
        "answer_relevancy",
        "context_precision",
        "context_recall",
    ]
    available = [m for m in metric_cols if m in df_all.columns]
    summary = df_all.groupby("pipeline")[available].mean().round(3)

    print("\n AVERAGE RAGAS SCORES")
    print("=" * 70)
    print(summary)
    print("=" * 70)

    if {"Naive", "Advanced"}.issubset(summary.index):
        lift = (summary.loc["Advanced"] - summary.loc["Naive"]).round(3)
        lift_pct = (
            (summary.loc["Advanced"] - summary.loc["Naive"])
            / summary.loc["Naive"].replace(0, 0.01)
            * 100
        ).round(1)
        print("\n ADVANCED vs NAIVE — Absolute Lift")
        print(lift)
        print("\n ADVANCED vs NAIVE — % Lift")
        print(lift_pct.astype(str) + " %")
    return summary


def save_bar_chart(summary: pd.DataFrame, outfile: Path) -> None:
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(10, 6))
    summary.T.plot(
        kind="bar",
        ax=ax,
        rot=15,
        color=["#C73E1D", "#2E86AB"],
        edgecolor="black",
        width=0.7,
    )
    ax.set_title(
        "Naive vs Advanced RAG — RAGAS Quality Metrics",
        fontsize=14,
        fontweight="bold",
    )
    ax.set_ylabel("Score (0 = bad, 1 = perfect)")
    ax.set_ylim(0, 1.05)
    ax.axhline(y=0.7, color="gray", linestyle="--", alpha=0.5, label="Production threshold")
    ax.legend(title="Pipeline", loc="lower right")
    ax.grid(axis="y", alpha=0.3)
    for c in ax.containers:
        ax.bar_label(c, fmt="%.2f", padding=3, fontsize=10)
    plt.tight_layout()
    plt.savefig(outfile, dpi=150)
    print(f"📊 Saved bar chart → {outfile}")


def parse_args() -> argparse.Namespace:
    p = argparse.ArgumentParser(description="RAGAS evaluation of Naive vs Advanced RAG")
    p.add_argument(
        "--judge",
        default="llama-3.1-8b-instant",
        help="Groq judge model (default: llama-3.1-8b-instant; try llama-3.3-70b-versatile for stricter grading)",
    )
    p.add_argument(
        "--outdir",
        default="results",
        help="Directory for CSV + PNG outputs (default: results/)",
    )
    p.add_argument(
        "--no-chart",
        action="store_true",
        help="Skip matplotlib bar chart (useful in headless CI)",
    )
    return p.parse_args()


def main() -> None:
    args = parse_args()
    outdir = Path(args.outdir)
    outdir.mkdir(parents=True, exist_ok=True)

    load_groq_key()
    print(f"🔧 Judge model: {args.judge}\n")

    # 1. Build both pipelines
    print("═" * 70)
    print(" Building Naive RAG …")
    print("═" * 70)
    naive = NaiveRAG.build()

    print("\n" + "═" * 70)
    print(" Building Advanced RAG …")
    print("═" * 70)
    advanced = AdvancedRAG.build()

    # 2. Generate answers on the golden set
    naive_rows = run_pipeline(naive, TEST_SET, "Naive RAG")
    adv_rows = run_pipeline(advanced, TEST_SET, "Advanced RAG")

    # 3. Get embeddings for RAGAS (same model as retrieval — cheap wrapper reuses cache)
    from langchain_huggingface import HuggingFaceEmbeddings
    embeddings = HuggingFaceEmbeddings(model_name=DEFAULT_EMBED_MODEL)

    # 4. Score both pipelines with RAGAS
    judge_llm, judge_embeddings, metrics = build_ragas_judge(args.judge, embeddings)
    df_naive = evaluate_pipeline(naive_rows, "Naive", judge_llm, judge_embeddings, metrics)
    df_adv = evaluate_pipeline(adv_rows, "Advanced", judge_llm, judge_embeddings, metrics)
    df_all = pd.concat([df_naive, df_adv], ignore_index=True)

    # 5. Print summary + save artefacts
    summary = print_summary(df_all)

    df_all.to_csv(outdir / "ragas_all_scores.csv", index=False)
    summary.to_csv(outdir / "ragas_summary.csv")
    print(f"\n💾 Saved raw scores → {outdir/'ragas_all_scores.csv'}")
    print(f"💾 Saved summary   → {outdir/'ragas_summary.csv'}")

    if not args.no_chart:
        save_bar_chart(summary, outdir / "ragas_comparison.png")

    print("\n✅ Evaluation complete")


if __name__ == "__main__":
    main()


## Step 2 — Install dependencies

Same known-good version matrix as the notebook. Takes ~90 s first time.

> If you see the RAGAS↔`ChatVertexAI` import error, **Runtime → Restart runtime** once
> after this cell finishes and re-run from Step 3 onwards.


In [ ]:
!pip install -q \
    groq python-dotenv \
    "langchain>=0.3,<0.4" "langchain-core>=0.3,<0.4" "langchain-community>=0.3,<0.4" \
    "langchain-groq>=0.3,<1.0" "langchain-text-splitters>=0.3,<0.4" "langchain-huggingface>=0.1,<0.4" \
    sentence-transformers faiss-cpu rank_bm25 pymupdf \
    "ragas>=0.2.10,<0.3" datasets pandas matplotlib

print("\n✅ Dependencies installed")


## Step 3 — Load your Groq key from Colab Secrets

The `common.load_groq_key()` helper already prefers `google.colab.userdata`,
so this cell is just a one-line trigger + status message.


In [ ]:
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
assert os.environ["GROQ_API_KEY"].startswith("gsk_"), \
    "❌ GROQ_API_KEY not set — open the 🔑 sidebar → Add secret named GROQ_API_KEY"
print("✅ GROQ_API_KEY loaded from Colab secrets")


## Step 4 — Smoke test (`common.py`) — verifies PDF + FAISS + Groq call all work

In [ ]:
!python common.py

## Step 5 — Run **Naive RAG** on 4 sample questions

Expected: the *authors* query fails (`Not found in document`) — dense embeddings can't
find proper nouns. That failure is the motivation for the Advanced pipeline next.


In [ ]:
!python naive_rag.py

## Step 6 — Run **Advanced RAG** on the same 4 questions

Expected: the *authors* query now succeeds — BM25 catches the exact name tokens,
RRF merges dense + sparse, and the cross-encoder lifts the correct chunk to rank 1.


In [ ]:
!python advanced_rag.py

## Step 7 — Full **RAGAS Evaluation** (Naive vs Advanced × 4 metrics)

Metrics: **Faithfulness · Answer Relevancy · Context Precision · Context Recall**.
Judge: Groq Llama-3.1-8B (free, generous quota). Add `--judge llama-3.3-70b-versatile`
for stricter grading (may hit free-tier daily cap).

Outputs are written to `results/`:
- `ragas_all_scores.csv` — per-question × per-metric
- `ragas_summary.csv` — per-pipeline averages
- `ragas_comparison.png` — bar chart


In [ ]:
!python evaluate.py --outdir results/

### 📊 Show the bar chart inline

In [ ]:
from IPython.display import Image, display
display(Image("results/ragas_comparison.png"))


### 📋 Peek at the raw scores

In [ ]:
import pandas as pd
df = pd.read_csv("results/ragas_all_scores.csv")
df


## Optional — import the pipelines directly in this notebook

Instead of `!python advanced_rag.py`, you can drive the classes from Python:


In [ ]:
import sys
sys.path.insert(0, "/content")   # so imports find common.py etc.

from advanced_rag import AdvancedRAG

rag = AdvancedRAG.build()
out = rag.query("What is multi-head attention?")
print("Pages cited:", out["pages_used"])
print("\nAnswer:\n", out["answer"])


## 🛟 Troubleshooting

| Symptom | Fix |
|---|---|
| `ImportError: cannot import name 'ChatVertexAI'` | **Runtime → Restart runtime** after Step 2, then re-run from Step 3 |
| `RateLimitError 429 ... TPD` | You hit Groq's free daily cap — wait ~15 min OR keep the 8B judge |
| Cross-encoder is slow | **Runtime → T4 GPU** and change `reranker_device="cpu"` to `"cuda"` in `advanced_rag.py` (Step 1 cell) |
| `assert GROQ_API_KEY` fails | Sidebar 🔑 → Add secret named `GROQ_API_KEY` → toggle **Notebook access** ON |
| First run downloads ~90 MB | HuggingFace MiniLM cache — one-off |

Docs: [RAG_Notebook_CellByCell.md](https://github.com/) · [RAG_Algorithms_DeepDive.md](https://github.com/) (from the workspace README).
